Import Lexer

In [191]:
from src.lexer import lexer
from __future__ import annotations
from enum import Enum
from abc import ABC, abstractmethod
#import time
import re
import datetime
import sys

AST

In [192]:
class NodeError(Exception):
    """Error for the nodes"""
    pass

In [193]:
class Node(ABC):
    """
    """
    def __init__(self):
        self.children: list[Node] = []
    
    def addChild(self, child: Node):
        """
        """
        if not isinstance(child, Node):
            raise NodeError("Node.addChild() child is not a Node")
        self.children.append(child)

    def removeChild(self, pos: int = 0):
        if not len(self.children) >= pos + 1:
            raise IndexError("Node.removeChild() pos is larger than self.children")
        del self.children[pos]

    @abstractmethod
    def build(self):
        """
        """
        pass

In [194]:
class TimeNode(Node):
    """
    """
    def __init__(self, contents: datetime.datetime):
        super().__init__()
        self.contents: datetime.datetime = contents
    
    def build(self):
        """
        """
        pass

In [195]:
class ActivityNode(Node):
    """
    """
    def __init__(self, contents: str):
        super().__init__()
        self.contents: str = contents
    
    def build(self):
        """
        """
        pass

In [196]:
class CommentNode(Node):
    """
    """
    def __init__(self, contents: str):
        super().__init__()
        self.contents: str = contents
    
    def build(self):
        """
        """
        pass

In [197]:
class AST:
    """
    """
    def __init__(self, root: Node):
        self.root: Node = root

Parser

In [198]:
class ParserState(Enum):
    """
    """
    NORMAL = 1
    ACTIVITY_COMMENT = 2

In [199]:
class ParserNoMatchError(Exception):
    """Error for the parser"""
    pass

class ParserModeError(Exception):
    """
    """
    pass

In [200]:
class ParserMode(ABC):
    """
    """
    def __init__(self, mode: ParserState):
        self.mode: ParserState = mode

    @abstractmethod
    def build(self, lexer_stack: list[lexer.Lexeme]) -> AST:
        """
        """
        pass

In [201]:
class ParserTimeMode(ParserMode):
    """
    """
    def __init__(self, mode: ParserState):
        super().__init__(mode)

    def _validateTime(self, source_str: str) -> datetime.datetime:
        """Get the times from a input_text string"""
        time_pattern = r"^[0-9]{1,2}:[0-9]{1,2}$"
        matches = re.match(time_pattern, source_str)
        if matches is None:
            raise ValueError("ParserTimeMode._validateTime() source_str is not a valid time")
        #return time.strptime(matches.group(), "%H:%M")
        return datetime.datetime.strptime(matches.group(), "%H:%M")

    def _validateDateTime(self, source_str: str) -> datetime.datetime:
        """Get the times from a input_text string"""
        time_pattern = r"^[0-9]{4}-[0-9]{2}-[0-9]{2} [0-9]{1,2}:[0-9]{1,2}$"
        matches = re.match(time_pattern, source_str)
        if matches is None:
            try:
                return self._validateTime(source_str)
            except:
                raise ValueError("ParserTimeMode._validateDateTime() source_str is not a valid time")
        return datetime.datetime.strptime(matches.group(), "%Y-%m-%d %H:%M")
    
    def build(self, lexer_stack: list[lexer.Lexeme]) -> AST:
        """
        """
        try:
            processed_time = self._validateDateTime(lexer_stack[0].text)
            return AST(TimeNode(processed_time))
        except:
            raise ParserModeError("ParserTimeMode.build() time is not valid")
        

In [202]:
class ParserSentenceMode(ParserMode):
    """
    """
    def __init__(self, mode: ParserState):
        super().__init__(mode)
    
    def build(self, lexer_stack: list[lexer.Lexeme]) -> AST:
        """
        """
        if lexer_stack[0].identifier is lexer.LexerTokenEnum.SENTENCE:
            return AST(ActivityNode(lexer_stack[0].text))
        if lexer_stack[0].identifier is lexer.LexerTokenEnum.COMMENT:
            return AST(CommentNode(lexer_stack[1].text))
        

In [203]:
class ParserActivityCommentMode(ParserMode):
    """
    """
    def __init__(self, mode: ParserState):
        super().__init__(mode)
    
    def build(self, lexer_stack: list[lexer.Lexeme]) -> AST:
        """
        """
        activity = ActivityNode(lexer_stack[0].text)
        activity.addChild(CommentNode(lexer_stack[2].text))
        return(AST(activity))

In [204]:
class ParserEOIFlag(Exception):
    """
    """
    pass

In [205]:
class ParserEOIMode(ParserMode):
    """
    """
    def __init__(self, mode: ParserState):
        super().__init__(mode)
    
    def build(self, lexer_stack: list[lexer.Lexeme]) -> AST:
        """
        """
        raise(ParserEOIFlag("End of Input reached"))
        

In [206]:
class ParserExpression:
    """
    """
    def __init__(self, parser_expression: list[lexer.LexerTokenEnum], parser_mode: ParserMode):
        self.parser_expression: list[lexer.LexerTokenEnum] = parser_expression
        self.parser_mode: ParserMode = parser_mode

    def _checkInd(self, lexer_token: str, exp_pos: int) -> bool:
        """Method to check individual characters of the text and the token"""
        if lexer_token is self.parser_expression[exp_pos]:
            return True
        return False

    def checkExp(self, lexer_stack: list[lexer.Lexeme]) -> tuple[ParserMode, int]:
        """Method to check if the text matches a token"""        
        if len(lexer_stack) < len(self.parser_expression):
            raise ParserNoMatchError("ParserExpression.checkExp() lexer_stack smaller than self.parser_expression")
        
        if len(lexer_stack) > 0 and len(lexer_stack) >= len(self.parser_expression):
            for exp_pos in range(len(self.parser_expression)):
                if not self._checkInd(lexer_stack[exp_pos].identifier, exp_pos):
                    raise ParserNoMatchError("ParserExpression.checkExp() no match")
            return (self.parser_mode, len(self.parser_expression))
            #return self.parser_mode().build(self.parser_expression[0:len(self.parser_expression)])
        raise ParserNoMatchError("ParserExpression.checkExp() no match")
    

In [207]:
class ParserExpressions:
    """Class to hold tokens for the lexer"""
    def __init__(self):
        self.exps: list[ParserExpression] = []
    
    def addExp(self, exp: ParserExpression):
        """Method to add tokens to the container"""
        if not isinstance(exp, ParserExpression):
            raise TypeError("ParserExpressions.addExp() token is not a ParserExpression")
        self.exps.append(exp)

In [208]:
at_at_mode = lexer.LexerToken(search_term = "@@", lexer_token = lexer.LexerTokenEnum.TIME)
at_less_mode = lexer.LexerToken(search_term = "@<", lexer_token = lexer.LexerTokenEnum.COMMENT)
greater_at_mode = lexer.LexerToken(search_term = ">@", lexer_token = lexer.LexerTokenEnum.ACTIVITY)

lexer_tokens = lexer.LexerTokens()
lexer_tokens.addToken(at_at_mode)
lexer_tokens.addToken(at_less_mode)
lexer_tokens.addToken(greater_at_mode)

test_lexer = lexer.Lexer(lexer_tokens)

In [209]:
time_exp = ParserExpression(
    [lexer.LexerTokenEnum.SENTENCE,
     lexer.LexerTokenEnum.TIME],
     ParserTimeMode(ParserState.NORMAL))
activity_exp = ParserExpression(
    [lexer.LexerTokenEnum.SENTENCE,
     lexer.LexerTokenEnum.ACTIVITY],
     ParserSentenceMode(ParserState.NORMAL))
activity_comment_exp = ParserExpression(
    [lexer.LexerTokenEnum.SENTENCE, 
     lexer.LexerTokenEnum.COMMENT,
     lexer.LexerTokenEnum.SENTENCE, 
     lexer.LexerTokenEnum.ACTIVITY],
     ParserActivityCommentMode(ParserState.ACTIVITY_COMMENT))
comment_exp = ParserExpression(
    [lexer.LexerTokenEnum.COMMENT,
     lexer.LexerTokenEnum.SENTENCE,
     lexer.LexerTokenEnum.END_OF_INPUT],
     ParserSentenceMode(ParserState.NORMAL))
end_of_input_exp = ParserExpression(
    [lexer.LexerTokenEnum.END_OF_INPUT],
     ParserEOIMode(ParserState.NORMAL)) ###

parser_exps = ParserExpressions()
parser_exps.addExp(time_exp)
parser_exps.addExp(activity_exp)
parser_exps.addExp(activity_comment_exp)
parser_exps.addExp(comment_exp)
parser_exps.addExp(end_of_input_exp)

In [210]:
class Parser:
    """
    """
    def __init__(self, lexer: lexer.Lexer, parser_exps: ParserExpressions):
        self.lexer: lexer.Lexer = lexer
        self.parser_exps: ParserExpressions = parser_exps
        self.parsed: list[AST] = []

    def _checkMatch(self):
        """
        """
        for exp in self.parser_exps.exps:
            try:
                return exp.checkExp(self.lexer.lexed)
            except:
                pass
        raise ParserNoMatchError("Parser._checkMatch() has no expression match")

    def reset(self):
        """
        """
        self.lexer.reset()
        self.ast = []

    def searchParser(self, src_txt: str):
        """
        """
        self.reset()
        self.lexer.searchLexer(src_txt)
        i = 0
        while i < 20: #True:
            try:
                matched = self._checkMatch()
                built_AST = matched[0].build(self.lexer.lexed[0:matched[1]])
                self.lexer.removeLexeme(matched[1])
                self.parsed.append(built_AST)
                if len(self.lexer.lexed) == 0:
                    break
            except ParserEOIFlag:
                break
            except Exception as e:
                sys.exit((
                    f"{e}\n"
                    f"{self.lexer.lexed[0:matched[1]]}"
                ))
            i = i + 1


In [211]:
test_str = "2026-04-24 07:29 @@ Bathroom >@ Reading Berserk @< Cool Chapter >@ @< Chapter 176"
#test_str = "2026-04-24 07:29 @@ Bathroom >@ Reading Berserk @< Cool Chapter >@"
test_parser = Parser(test_lexer, parser_exps)
test_parser.searchParser(test_str)

In [212]:
test_parser.parsed

In [215]:
for i in test_parser.parsed:
    print(i.root.contents)
    for x in i.root.children:
        print("\t " + x.contents)

2026-04-24 07:29:00
Bathroom
Reading Berserk
	 Cool Chapter
Chapter 176
